# Exploración inicial del dataset — MIMIC-IV 3.0

Primer paso del pipeline: caracterización agregada de MIMIC-IV 3.0 antes de construir la
cohorte Sepsis-3 (notebook `02`). Se revisan estancias en UCI, demografía, admisiones, los
*item IDs* de constantes vitales y de laboratorio relevantes para SOFA/Sepsis-3, la
microbiología como *proxy* de infección sospechada, y la **cobertura** de cada modalidad.

> **Privacidad.** Todas las salidas son estadísticas agregadas. En ningún momento se
> muestran datos de pacientes individuales, por lo que este notebook es apto para el
> repositorio público.

**Requisito:** MIMIC-IV 3.0 descargado localmente en `~/mimic-iv-3.0/` (acceso vía
PhysioNet + certificación CITI; los datos nunca se redistribuyen).

In [ ]:
import polars as pl
from pathlib import Path

MIMIC_PATH = Path.home() / "mimic-iv-3.0"
HOSP = MIMIC_PATH / "hosp"
ICU  = MIMIC_PATH / "icu"

def header(title: str):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print('='*60)

## 1 — Estancias en UCI (ICU stays)

In [ ]:
header("1. ICU STAYS")

stays = pl.read_csv(ICU / "icustays.csv.gz")
print(f"Total ICU stays:      {len(stays):,}")
print(f"Unique patients:      {stays['subject_id'].n_unique():,}")
print(f"Unique hadm_id:       {stays['hadm_id'].n_unique():,}")

los = (pl.col("outtime").str.to_datetime() - pl.col("intime").str.to_datetime()).dt.total_hours()
stays = stays.with_columns(los.alias("los_hours"))

print("\nLOS (horas) — filtros por duración mínima de estancia:")
stays_24h = stays.filter(pl.col("los_hours") >= 24)
print(f"  N >= 24h:  {len(stays_24h):,}  ({100*len(stays_24h)/len(stays):.1f}%)")
stays_48h = stays.filter(pl.col("los_hours") >= 48)
print(f"  N >= 48h:  {len(stays_48h):,}  ({100*len(stays_48h)/len(stays):.1f}%)")

print("\nDistribución LOS (horas):")
print(stays["los_hours"].describe())

## 2 — Demografía de pacientes

In [ ]:
header("2. PATIENTS")

patients = pl.read_csv(HOSP / "patients.csv.gz")
print(f"Total unique patients (hosp): {patients['subject_id'].n_unique():,}")
print("\nGénero (gender):")
print(patients["gender"].value_counts().sort("count", descending=True))
print(f"\nEdad media (anchor_age): {patients['anchor_age'].mean():.1f} ± {patients['anchor_age'].std():.1f}")
print(f"Rango: {patients['anchor_age'].min()} – {patients['anchor_age'].max()}")

## 3 — Admisiones y mortalidad hospitalaria

In [ ]:
header("3. ADMISSIONS")

adm = pl.read_csv(HOSP / "admissions.csv.gz")
print(f"Total admissions: {len(adm):,}")
print("\nMortalidad hospitalaria (hospital_expire_flag):")
print(adm["hospital_expire_flag"].value_counts().sort("hospital_expire_flag"))

## 4 — Constantes vitales: *item IDs* clave (`chartevents`)

Localizamos en `d_items` los identificadores de las constantes relevantes para sepsis, que
alimentarán la extracción de series temporales del notebook `03`.

In [ ]:
header("4. CHARTEVENTS — item IDs de vitales clave")

d_items = pl.read_csv(ICU / "d_items.csv.gz", infer_schema_length=10000)
vital_keywords = ["heart rate", "blood pressure", "respiratory rate",
                  "temperature", "oxygen saturation", "spo2", "gcs",
                  "systolic", "diastolic", "mean arterial"]
mask = pl.lit(False)
for kw in vital_keywords:
    mask = mask | pl.col("label").str.to_lowercase().str.contains(kw)
vitals_items = d_items.filter(mask).select(["itemid", "label", "unitname"])
print(f"Items de vitales encontrados: {len(vitals_items)}")
print(vitals_items.head(30).to_pandas().to_string(index=False))

## 5 — Laboratorio: *item IDs* relevantes para SOFA/Sepsis-3 (`labevents`)

In [ ]:
header("5. LAB ITEMS — relevantes para SOFA/Sepsis-3")

d_labitems = pl.read_csv(HOSP / "d_labitems.csv.gz")
lab_keywords = ["creatinine", "bilirubin", "platelet", "lactate",
                "white blood", "wbc", "pao2", "fio2", "sodium",
                "potassium", "glucose", "hemoglobin", "troponin"]
mask_lab = pl.lit(False)
for kw in lab_keywords:
    mask_lab = mask_lab | pl.col("label").str.to_lowercase().str.contains(kw)
sofa_labs = d_labitems.filter(mask_lab).select(["itemid", "label", "fluid", "category"])
print(f"Lab items SOFA/sepsis: {len(sofa_labs)}")
print(sofa_labs.to_pandas().to_string(index=False))

## 6 — Microbiología: *proxy* de infección sospechada

Los cultivos (`microbiologyevents`) son uno de los dos componentes del criterio de Angus
(cultivo + antibiótico IV) que define la infección sospechada en el notebook `02`.

In [ ]:
header("6. MICROBIOLOGYEVENTS — distribución de cultivos")

micro = pl.read_csv(HOSP / "microbiologyevents.csv.gz",
                    columns=["subject_id", "hadm_id", "spec_type_desc", "org_name"])
print(f"Total registros microbiología: {len(micro):,}")
print(f"Pacientes únicos con cultivo:  {micro['subject_id'].n_unique():,}")
print("\nTop 10 tipos de muestra:")
print(micro["spec_type_desc"].value_counts().sort("count", descending=True).head(10))

## 7 — Cobertura: estancias con datos de laboratorio

A partir de aquí evaluamos qué fracción de las estancias tiene realmente cada modalidad
disponible: es lo que acota el tamaño de la cohorte utilizable. Solo leemos la columna
`hadm_id` de `labevents` (>120 M filas), por lo que la lectura puede tardar ~30 s.

In [ ]:
header("7. COBERTURA: ICU stays con datos de laboratorio")

print("Leyendo labevents hadm_id (puede tardar ~30s)...")
lab_hadm = (
    pl.read_csv(HOSP / "labevents.csv.gz", columns=["hadm_id"])
    .drop_nulls()
    .with_columns(pl.col("hadm_id").cast(pl.Int64))
)
lab_hadm_list = lab_hadm["hadm_id"].unique().to_list()

stays_with_labs = stays.filter(pl.col("hadm_id").cast(pl.Int64).is_in(lab_hadm_list))
pct = 100 * len(stays_with_labs) / len(stays)
print(f"ICU stays con al menos 1 lab result: {len(stays_with_labs):,} ({pct:.1f}%)")

stays_24h_labs = stays_24h.filter(pl.col("hadm_id").cast(pl.Int64).is_in(lab_hadm_list))
pct_24h = 100 * len(stays_24h_labs) / len(stays_24h)
print(f"  → De los stays ≥24h: {len(stays_24h_labs):,} ({pct_24h:.1f}%)")

## 8 — Cobertura: estancias con constantes vitales (`chartevents`)

Análogo al anterior, leyendo solo `stay_id` de `chartevents` (>330 M filas); ~60 s.

In [ ]:
header("8. COBERTURA: ICU stays con chartevents (vitales)")

print("Leyendo chartevents stay_id (puede tardar ~60s)...")
chart_stays = (
    pl.read_csv(ICU / "chartevents.csv.gz", columns=["stay_id"])
    .drop_nulls()
    .with_columns(pl.col("stay_id").cast(pl.Int64))
)
chart_stay_list = chart_stays["stay_id"].unique().to_list()

stays_with_charts = stays.filter(pl.col("stay_id").cast(pl.Int64).is_in(chart_stay_list))
pct2 = 100 * len(stays_with_charts) / len(stays)
print(f"ICU stays con al menos 1 chart event: {len(stays_with_charts):,} ({pct2:.1f}%)")

stays_24h_charts = stays_24h.filter(pl.col("stay_id").cast(pl.Int64).is_in(chart_stay_list))
pct2_24h = 100 * len(stays_24h_charts) / len(stays_24h)
print(f"  → De los stays ≥24h: {len(stays_24h_charts):,} ({pct2_24h:.1f}%)")

## 9 — Cobertura combinada: cohorte base potencial

Estancias ≥24 h que disponen **simultáneamente** de laboratorio y constantes vitales. Es la
cohorte base sobre la que el notebook `02` aplicará el filtro Sepsis-3.

In [ ]:
header("9. COBERTURA COMBINADA: stays ≥24h con labs Y vitales")

stays_both = stays_24h.filter(
    pl.col("hadm_id").cast(pl.Int64).is_in(lab_hadm_list) &
    pl.col("stay_id").cast(pl.Int64).is_in(chart_stay_list)
)
pct_both = 100 * len(stays_both) / len(stays_24h)
print(f"Stays ≥24h con labs Y chartevents: {len(stays_both):,} ({pct_both:.1f}% de los ≥24h)")
print(f"Pacientes únicos en esta cohorte:   {stays_both['subject_id'].n_unique():,}")

header("COMPLETADO — cohorte potencial antes de filtro Sepsis-3")
print(f"Cohorte base estimada: {len(stays_both):,} stays / {stays_both['subject_id'].n_unique():,} pacientes")
print("\nNinguna salida contiene datos de pacientes individuales.")